In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from pyod.models.deep_svdd import DeepSVDD
import sys
import scrapbook as sb

sys.path.append('..')
from utils import reshape_to_numpy, interpolate_missing_values, denoise_data, compute_spectrograms, time_avg_pooling

In [2]:
seed = 1
sampling_rate = 10
nperseg=64
noverlap=32
accel_cutoff = 0.4
accel_order = 30
gyro_cutoff = 0.6
gyro_order = 20
batch_size = 32
epochs = 100
hidden_neurons = [128, 64, 32]
n_splits = 5

In [3]:
# Parameters
seed = 26


In [4]:
rng = np.random.RandomState(seed)

In [5]:
data = pd.read_parquet("../data/GBG500.parquet")
data

,ride_id,time_index,ax,ay,az,rx,ry,rz
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,0,-2.512796,-9.385012,-1.053078,-0.009155,0.009155,-0.201416
1,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,100,-2.491268,-9.385012,-1.079390,-0.036621,0.027465,-0.155639
2,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,200,-2.534324,-9.382022,-1.030952,0.036621,0.027465,-0.073242
3,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,300,-2.488278,-9.383218,-1.091948,-0.045776,0.036621,-0.183105
4,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,400,-2.483494,-9.382022,-1.100320,-0.045776,0.027465,-0.192260
...,...,...,...,...,...,...,...,...
1529542,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241300,0.459862,-9.140430,-3.318900,1.556396,-0.091552,0.274658
1529543,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241400,0.455676,-9.140430,-3.321292,1.583861,-0.119018,0.274658
1529544,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241500,0.459862,-9.139832,-3.317704,1.583861,-0.109863,0.274658
1529545,ffb6f2f5dc6ba768d88dc0bc1af4cad235526aa9e24168...,241600,0.456274,-9.142224,-3.321292,1.574707,-0.137329,0.283813


In [6]:
labels = pd.read_csv("../data/GBG500_labels.csv")
labels.columns = labels.columns.str.lower()
ride_order_df = pd.DataFrame({"ride_id": data["ride_id"].unique()})
labels_sorted = ride_order_df.merge(
    labels,
    on="ride_id",
    how="left"
)
labels_sorted

,ride_id,label
0,005043f1a7520cebf1e17fd166a00a3190e2433530f236...,Safe
1,00d223cb7aecc9c0cc5871e32a6c45027a068eba07c572...,Reckless
2,02288a4aeca044203394e982e76b87818021dea2b34df9...,Safe
3,0236bdcf13d473ea24d97f5eaeea459f257dffac005694...,Bad weather
4,0238e5dd85143f1b6c59b200bc32f14361b8fc84c51a87...,Safe
...,...,...
495,fe5d7f84d692bbccad8bd566a3ad5c3108fa9f90acd671...,Safe
496,fe67169632306d4668b2affedef510df2cb43e2fb41220...,Safe
497,fee44667fdc8ab4995c702fc3bd36178de0b1ca2c64687...,Safe
498,ff79b2e945b93c2a5efc3a06365267b3a160e7849ba57d...,Safe


In [7]:
data_np = reshape_to_numpy(
    data,
    features = ["ax", "ay", "az", "rx", "ry", "rz"],
    max_timestamps = 4800
)

data_np_clean = interpolate_missing_values(
    data_np,
    method='linear',
    limit=None
)

data_np_clean = denoise_data(
    data=data_np_clean,
    accel_indices=[0, 1, 2],
    gyro_indices=[3, 4, 5],
    accel_cutoff=accel_cutoff,
    accel_order=accel_order,
    gyro_cutoff=gyro_cutoff,
    gyro_order=gyro_order,
)

In [8]:
# Compute spectrograms for all rides
spectrograms_array = compute_spectrograms(data_np_clean, sampling_rate, nperseg, noverlap)
spectrograms_array.shape

(500, 149, 6, 33)

In [9]:
# Aggregate spectrograms using time-averaged pooling
X_feat = time_avg_pooling(spectrograms_array)
X_feat.shape

(500, 198)

In [10]:
# 5-fold stratified CV with Deep SVDD
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=rng.randint(1000))
y_true = (labels_sorted['label'] == 'Reckless').astype(int).values
anomaly_scores = np.full(len(y_true), np.nan)

for fold, (train_idx, test_idx) in enumerate(skf.split(X_feat, y_true)):
    sc = StandardScaler()
    X_train = sc.fit_transform(X_feat[train_idx])
    X_test = sc.transform(X_feat[test_idx])

    np.random.seed(rng.randint(1000))
    model = DeepSVDD(
        n_features=X_train.shape[1],
        hidden_neurons=hidden_neurons,
        epochs=epochs,
        batch_size=batch_size,
        random_state=rng.randint(1000),
    )
    model.fit(X_train)
    anomaly_scores[test_idx] = model.decision_function(X_test)
    print(f"Fold {fold+1}/{n_splits} done")

Epoch 1/100, Loss: 1.5760063007473946
Epoch 2/100, Loss: 1.4101946130394936
Epoch 3/100, Loss: 1.5183211900293827
Epoch 4/100, Loss: 1.4865358918905258
Epoch 5/100, Loss: 1.493664439767599
Epoch 6/100, Loss: 1.4633923061192036
Epoch 7/100, Loss: 1.5876053608953953
Epoch 8/100, Loss: 1.5735607333481312
Epoch 9/100, Loss: 1.4890321977436543
Epoch 10/100, Loss: 1.633191667497158
Epoch 11/100, Loss: 1.4973999485373497
Epoch 12/100, Loss: 1.648085094988346
Epoch 13/100, Loss: 1.6505354791879654
Epoch 14/100, Loss: 1.452618569135666
Epoch 15/100, Loss: 1.5776635892689228


Epoch 16/100, Loss: 1.4890705309808254
Epoch 17/100, Loss: 1.616627987474203
Epoch 18/100, Loss: 1.5546594373881817
Epoch 19/100, Loss: 1.5930058173835278
Epoch 20/100, Loss: 1.6105603612959385
Epoch 21/100, Loss: 1.4883886575698853
Epoch 22/100, Loss: 1.4596171490848064
Epoch 23/100, Loss: 1.6688117422163486
Epoch 24/100, Loss: 1.3869864270091057
Epoch 25/100, Loss: 1.692107405513525
Epoch 26/100, Loss: 1.6332714706659317
Epoch 27/100, Loss: 1.7315750867128372
Epoch 28/100, Loss: 1.642834085971117
Epoch 29/100, Loss: 1.472186852246523
Epoch 30/100, Loss: 1.3427846245467663


Epoch 31/100, Loss: 1.485952291637659
Epoch 32/100, Loss: 1.5585597194731236
Epoch 33/100, Loss: 1.5207415781915188
Epoch 34/100, Loss: 1.4317724592983723
Epoch 35/100, Loss: 1.6300997771322727
Epoch 36/100, Loss: 1.677630040794611
Epoch 37/100, Loss: 1.5124827660620213
Epoch 38/100, Loss: 1.2873146794736385
Epoch 39/100, Loss: 1.5419403426349163
Epoch 40/100, Loss: 1.4744836054742336
Epoch 41/100, Loss: 1.6042334847152233
Epoch 42/100, Loss: 1.6105320453643799
Epoch 43/100, Loss: 1.499986957758665
Epoch 44/100, Loss: 1.5139587670564651
Epoch 45/100, Loss: 1.5602581016719341


Epoch 46/100, Loss: 1.5215534642338753
Epoch 47/100, Loss: 1.6715049147605896
Epoch 48/100, Loss: 1.4982102364301682
Epoch 49/100, Loss: 1.4798654094338417
Epoch 50/100, Loss: 1.5051529370248318
Epoch 51/100, Loss: 1.4769093096256256
Epoch 52/100, Loss: 1.426700048148632
Epoch 53/100, Loss: 1.4889186322689056
Epoch 54/100, Loss: 1.6488387361168861
Epoch 55/100, Loss: 1.5594786629080772
Epoch 56/100, Loss: 1.5239310748875141
Epoch 57/100, Loss: 1.5796367414295673
Epoch 58/100, Loss: 1.5091656148433685
Epoch 59/100, Loss: 1.5118295811116695
Epoch 60/100, Loss: 1.4665532857179642


Epoch 61/100, Loss: 1.6830243207514286
Epoch 62/100, Loss: 1.5878595560789108
Epoch 63/100, Loss: 1.540950920432806
Epoch 64/100, Loss: 1.5356835722923279
Epoch 65/100, Loss: 1.568246517330408
Epoch 66/100, Loss: 1.5857978723943233
Epoch 67/100, Loss: 1.4317760095000267
Epoch 68/100, Loss: 1.6616735272109509
Epoch 69/100, Loss: 1.3928481861948967
Epoch 70/100, Loss: 1.6813493147492409
Epoch 71/100, Loss: 1.7034926116466522
Epoch 72/100, Loss: 1.4044187739491463
Epoch 73/100, Loss: 1.5297305919229984
Epoch 74/100, Loss: 1.4731372222304344
Epoch 75/100, Loss: 1.5751546248793602


...

Epoch 23/100, Loss: 3.002299524843693
Epoch 24/100, Loss: 3.0610792338848114
Epoch 25/100, Loss: 2.9687558114528656
Epoch 26/100, Loss: 3.0065947771072388
Epoch 27/100, Loss: 3.1212934628129005
Epoch 28/100, Loss: 3.212507799267769
Epoch 29/100, Loss: 3.097028322517872
Epoch 30/100, Loss: 2.928538478910923
Epoch 31/100, Loss: 2.9397723376750946
Epoch 32/100, Loss: 2.993879556655884
Epoch 33/100, Loss: 3.3094514831900597
Epoch 34/100, Loss: 3.153268724679947
Epoch 35/100, Loss: 3.1296671107411385
Epoch 36/100, Loss: 3.1417132169008255
Epoch 37/100, Loss: 2.970839485526085
Epoch 38/100, Loss: 3.016925446689129


Epoch 39/100, Loss: 3.0871793180704117
Epoch 40/100, Loss: 2.9706321358680725
Epoch 41/100, Loss: 3.021490439772606
Epoch 42/100, Loss: 2.676651395857334
Epoch 43/100, Loss: 2.9706802740693092
Epoch 44/100, Loss: 3.64944801479578
Epoch 45/100, Loss: 3.009517341852188
Epoch 46/100, Loss: 2.8534215837717056
Epoch 47/100, Loss: 2.5237711369991302
Epoch 48/100, Loss: 3.0215930864214897
Epoch 49/100, Loss: 2.805106222629547
Epoch 50/100, Loss: 3.0218217745423317
Epoch 51/100, Loss: 2.9715711548924446
Epoch 52/100, Loss: 3.1917323246598244
Epoch 53/100, Loss: 2.8017403408885
Epoch 54/100, Loss: 3.0803227722644806
Epoch 55/100, Loss: 2.955692395567894


Epoch 56/100, Loss: 2.906125947833061
Epoch 57/100, Loss: 2.965343862771988
Epoch 58/100, Loss: 2.9218457639217377
Epoch 59/100, Loss: 3.2209587693214417
Epoch 60/100, Loss: 3.102122724056244
Epoch 61/100, Loss: 3.0384407490491867
Epoch 62/100, Loss: 2.9603912234306335
Epoch 63/100, Loss: 2.849942147731781
Epoch 64/100, Loss: 2.843621276319027
Epoch 65/100, Loss: 2.6417638286948204
Epoch 66/100, Loss: 2.969975098967552
Epoch 67/100, Loss: 2.902518130838871
Epoch 68/100, Loss: 3.1087865382432938
Epoch 69/100, Loss: 3.1249598041176796
Epoch 70/100, Loss: 2.839758314192295
Epoch 71/100, Loss: 3.1386661753058434
Epoch 72/100, Loss: 2.673202730715275
Epoch 73/100, Loss: 3.0130535513162613


Epoch 74/100, Loss: 3.5693419873714447
Epoch 75/100, Loss: 4.026500716805458
Epoch 76/100, Loss: 2.9535139724612236
Epoch 77/100, Loss: 2.908988893032074
Epoch 78/100, Loss: 3.1252824664115906
Epoch 79/100, Loss: 2.907545745372772
Epoch 80/100, Loss: 2.921198569238186
Epoch 81/100, Loss: 3.043905258178711
Epoch 82/100, Loss: 3.0331126675009727
Epoch 83/100, Loss: 2.9983664751052856
Epoch 84/100, Loss: 3.200675390660763
Epoch 85/100, Loss: 3.025908149778843
Epoch 86/100, Loss: 2.9589397311210632
Epoch 87/100, Loss: 2.9790817350149155
Epoch 88/100, Loss: 3.1162178069353104
Epoch 89/100, Loss: 3.185323715209961
Epoch 90/100, Loss: 3.176822856068611


Epoch 91/100, Loss: 2.8927143067121506
Epoch 92/100, Loss: 3.0754183754324913
Epoch 93/100, Loss: 2.905463181436062
Epoch 94/100, Loss: 2.9743983820080757
Epoch 95/100, Loss: 2.8917686119675636
Epoch 96/100, Loss: 3.122270978987217
Epoch 97/100, Loss: 3.1170518696308136
Epoch 98/100, Loss: 3.67029095441103
Epoch 99/100, Loss: 3.034867450594902
Epoch 100/100, Loss: 3.0662756636738777
Fold 5/5 done


In [11]:
ap = average_precision_score(y_true, anomaly_scores)
print(f"Deep SVDD AP (5-fold CV) = {ap:.4f}")
sb.glue("GBG500_ap_spectral_deep_svdd", float(ap))

Deep SVDD AP (5-fold CV) = 0.6450
